# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.


## Solution - Approach Followed :

1. **Initialization:** Configure OpenAI, Groq, and local Ollama clients securely using environment variables.
2. **Model Routing:** Stream text responses from either GPT-4o-mini or Llama 3.2 based on user selection, applying an expert educator system prompt.
3. **Voice Integration:** Transcribe microphone input via Groq Whisper, summarize text for spoken delivery, and return raw audio bytes via Groq TTS.


In [1]:
# imports
import os
import io
import time
import tempfile
from typing import List, Dict, Generator, Any
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# constants

MODEL_GPT = "gpt-4o-mini"
MODEL_GPT_OPEN_SOURCE = "gpt-oss"
MODEL_LLAMA = "llama3.2"
MODEL_LLAMA_BASE_URL = os.getenv("MODEL_BASE_URL", "http://localhost:11434/v1")


SYSTEM_PROMPT = """
You are an expert technical educator.

Answer the user's technical question clearly, accurately, and concisely.

- Explain technical terms when necessary.
- Break complex concepts into simple parts.
- For code, explain what it does and why it works.
- Use examples when helpful.
- Do not invent information. If uncertain, say so.
- Do not give a long answer unless the user asks for one.
- Write explanations in clear, natural language suitable for both reading and listening.
"""

In [3]:
# set up environment
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if (
    OPENAI_API_KEY
    and len(OPENAI_API_KEY) > 10
    and OPENAI_API_KEY.startswith("sk-proj-")
):
    print("OPENAI_API_KEY looks good so far.")
else:
    print(
        "There might be a problem with your API key? Please visit the troubleshooting notebook!"
    )


if GROQ_API_KEY and len(GROQ_API_KEY) > 10:
    print(f"Groq API Key exists and begins {GROQ_API_KEY[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OPENAI_API_KEY looks good so far.
Groq API Key exists and begins gsk_


In [4]:
ollama = OpenAI(api_key=os.getenv("OLLAMA_API_KEY", "ollama"), base_url=os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434/v1"))
gemini = OpenAI(api_key=os.getenv("GEMINI_API_KEY", "gemini"), base_url=os.getenv("GEMINI_BASE_URL", "http://127.0.0.1:11434/v1"))
openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY", "chatgpt"), base_url=os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:11434/v1"))
groq = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

## Run Ollama runs Ollama models locally

In [5]:
def stream_ollama(message: str, history: List[Dict[str, str]]) -> Generator[str, None, None]:
    """Generates a streaming response from the local Ollama model based on the user message and conversation history."""
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *history,
        {"role": "user", "content": message},
    ]

    try:
        stream = ollama.chat.completions.create(
            model=MODEL_LLAMA,
            messages=messages, # type: ignore
            stream=True,
        )

        response = ""
        for chunk in stream:
            response += chunk.choices[0].delta.content or ""
            yield response
    except Exception as e:
        yield f"Error connecting to Ollama: {str(e)}"

In [6]:
# gr.ChatInterface(fn=stream_ollama, type="messages").launch()

## Stream responses from GPT model

In [13]:
def stream_gpt(message: str, history: List[Dict[str, str]]) -> Generator[str, None, None]:
    """Runs GPT model via OpenAI client and yields chunks, with error fallback."""
    formatted_history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *formatted_history,
        {"role": "user", "content": message},
    ]

    try:
        stream = openai.chat.completions.create(
            model=MODEL_GPT_OPEN_SOURCE,
            messages=messages, # type: ignore
            stream=True,
        )

        response = ""
        for chunk in stream:
            # Safely handle empty chunks
            content = chunk.choices[0].delta.content
            if content:
                response += content
                yield response
                
    except Exception as e:
        yield f"**Error connecting to GPT:** {str(e)}"

## Route user query to the selected model

In [8]:
def stream_model(message: str, history: List[Dict[str, str]], model: str) -> Generator[str, None, None]:
    """Routes the input to the appropriate model function and yields the response in chunks."""
    if model == "GPT":
        result = stream_gpt(message, history)
    elif model == "Ollama":
        result = stream_ollama(message, history)
    else:
        raise ValueError("Unknown model")
    yield from result

## Prepare LLM text for conversational audio playback

In [9]:
def create_voice_response(text: str) -> str:
    """Parses and summarizes LLM-generated technical text into a conversational format suitable for audio playback."""
    messages = [
        {
            "role": "system",
            "content": """
You convert technical answers into natural spoken responses.

Make the response conversational and natural, like a helpful human
technical tutor.

Do not read Markdown syntax, headings, bullet points, or formatting aloud.

Keep the response short and concise.
Summarize the most important points in approximately 2 to 4 sentences.
""",
        },
        {"role": "user", "content": text},
    ]

    try:
        response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error generating voice summary: {e}")
        return "I encountered an error generating the voice response."

## Audio Processing: Text-to-Speech and Speech-to-Text

In [10]:
def text_to_speech(text: str) -> bytes | None:
    """Generates speech based on text using Groq and returns raw audio bytes."""
    if not text:
        return None

    try:
        response = groq.audio.speech.create(
            model="canopylabs/orpheus-v1-english",
            voice="troy",
            input=text,
            response_format="wav",
        )

        # Return the raw audio bytes directly to Gradio
        # No disk I/O, no temporary files, no cleanup needed!
        return response.content

    except Exception as e:
        print(f"Audio Generation Error: {str(e)}")
        return None

def transcribe_audio(audio_filepath: str) -> str:
    """Creates text based on the audio input, returning an empty string on failure."""
    if not audio_filepath:
        return ""

    try:
        with open(audio_filepath, "rb") as audio_file:
            transcription = groq.audio.transcriptions.create(
                file=audio_file, model="whisper-large-v3-turbo"
            )
        return transcription.text.strip()

    except Exception as e:
        print(f"Transcription Error: {str(e)}")
        return "Error: Could not transcribe audio."

## Master chat function to manage state and UI updates

In [11]:
def chat(message: str, history: List[Dict[str, str]], model: str) -> Generator[tuple, None, None]:
    """Manages the chat flow by updating history, streaming the text response from the selected model, and appending the generated audio."""
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": ""})

    yield history, None

    if model == "GPT":
        generator = stream_gpt(message, history[:-2])
    elif model == "Ollama":
        generator = stream_ollama(message, history[:-2])
    else:
        raise ValueError("Unknown model")

    full_response = ""

    for response in generator:
        full_response = response
        history[-1]["content"] = response
        yield history, None

    audio = text_to_speech(create_voice_response(full_response))
    yield history, audio

## Define and launch the Gradio User Interface

In [12]:
with gr.Blocks() as ui:
    model_selector = gr.Dropdown(["GPT", "Ollama"], value="GPT", label="Select Model")

    chatbot = gr.Chatbot(height=500, type="messages")

    audio_output = gr.Audio(label="AI Voice", autoplay=True)

    with gr.Row():
        message = gr.Textbox(label="Ask a technical question", scale=8)
        mic_input = gr.Audio(
            sources=["microphone"], type="filepath", show_label=False, scale=1
        )

    # TEXT INPUT
    message.submit(
        chat, inputs=[message, chatbot, model_selector], outputs=[chatbot, audio_output]
    )

    # MIC INPUT
    mic_input.stop_recording(transcribe_audio, inputs=mic_input, outputs=message).then(
        chat, inputs=[message, chatbot, model_selector], outputs=[chatbot, audio_output]
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
